# Etapa 1 · Extração do PLD horário (CCEE) para o Neon

Este notebook faz:

1. Lê configuração (`config.yaml`) e credencial (variável de ambiente, `.env` ou prompt). Nada fixo no código.
2. Carrega o banco de origem (`campo_real_energia_postgres.sql`) no Neon, se ainda não estiver carregado.
3. Cria o schema `externo`: **log de execuções**, **raw** (dado como chegou, em `jsonb`) e **tratada** (`pld_horario`), mais a tabela de rejeitados.
4. Extrai o PLD horário de 2025 e 2026 pela API CKAN da CCEE, com paginação, retry, tratamento de 403/429/timeout/resposta vazia/mudança de colunas.
5. Trata a raw para a tabela `externo.pld_horario` (grão: **submercado × hora**).
6. Verifica completude e mostra o log.

In [ ]:
%pip install -q psycopg2-binary pyyaml requests curl_cffi pandas

## 1. Configuração

In [2]:
import os, re, time, json, getpass, unicodedata
from pathlib import Path
from datetime import datetime, timezone

import yaml, requests, pandas as pd, psycopg2
from psycopg2.extras import execute_values, Json

CONFIG_PATH = Path("config.yaml")

CONFIG_PADRAO = """
banco:
  variavel_ambiente: NEON_DATABASE_URL
  arquivo_env: .env
  origem_sql: campo_real_energia_postgres.sql
  recarregar_origem: false     
ccee:
  base_url: https://dadosabertos.ccee.org.br/api/3/action
  dataset_id: pld_horario
  anos: [2025, 2026]
  resource_ids: {}            
  page_size: 5000
http:
  backends: [curl_cffi, requests]  
  impersonate: chrome124
  user_agent: "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
  accept_language: "pt-BR,pt;q=0.9"
  timeout: 60
  max_tentativas: 5
  backoff_segundos: 2
  pausa_entre_paginas: 0.5
tratamento:
  hora_base: auto              
  campos:                      
    submercado: [SUBMERCADO, COD_SUBMERCADO, SUBMERCADO_NOME]
    pld: [PLD_HORA, PLD, VALOR_PLD, PRECO]
    data_hora: [DATA_HORA, DATAHORA, DATA_HORA_INICIO]
    dia: [DIA, DATA]
    hora: [HORA, HORA_INICIO]
    mes_referencia: [MES_REFERENCIA, MES_REF]
"""

def mesclar(padrao, atual):
    """Valores do arquivo vencem; o que faltar vem do padrão."""
    for k, v in (atual or {}).items():
        if isinstance(v, dict) and isinstance(padrao.get(k), dict):
            mesclar(padrao[k], v)
        else:
            padrao[k] = v
    return padrao

cfg = yaml.safe_load(CONFIG_PADRAO)
if CONFIG_PATH.exists():
    atual = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8")) or {}
    faltando = [k for k in cfg if k not in atual]
    cfg = mesclar(cfg, atual)
    print("config.yaml existente lido." + (f" Seções completadas com o padrão: {faltando}" if faltando else ""))
else:
    CONFIG_PATH.write_text(CONFIG_PADRAO.strip() + "\n", encoding="utf-8")
    print("config.yaml criado com os valores padrão.")
B, C, H, T = cfg["banco"], cfg["ccee"], cfg["http"], cfg["tratamento"]
print("anos:", C["anos"], "| backends:", H["backends"])

config.yaml existente lido. Seções completadas com o padrão: ['banco', 'tratamento']
anos: [2025, 2026] | backends: ['curl_cffi', 'requests']


## 2. Conexão com o Neon e banco de origem

In [3]:
def ler_url_banco():
    var = B["variavel_ambiente"]
    if os.environ.get(var):
        return os.environ[var]
    env = Path(B["arquivo_env"])
    if env.exists():
        for linha in env.read_text(encoding="utf-8").splitlines():
            linha = linha.strip()
            if linha and not linha.startswith("#") and "=" in linha:
                k, v = linha.split("=", 1)
                if k.strip() == var:
                    return v.strip().strip('"').strip("'")
    return getpass.getpass(f"Cole a connection string do Neon ({var}): ")

def conectar():
    # connect_timeout alto: Não deixar o Neon dormir
    return psycopg2.connect(ler_url_banco(), connect_timeout=30,
                            keepalives=1, keepalives_idle=30, keepalives_interval=10, keepalives_count=5)

conn = conectar()
with conn.cursor() as cur:
    cur.execute("SELECT version()")
    print(cur.fetchone()[0])

PostgreSQL 18.6 (ac70818) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [5]:
def origem_carregada(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT to_regclass('campo_real_energia.medicao_geracao')")
        if cur.fetchone()[0] is None:
            return False
        cur.execute("SELECT COUNT(*) FROM campo_real_energia.medicao_geracao")
        return cur.fetchone()[0] > 0

if B["recarregar_origem"] or not origem_carregada(conn):
    sql = Path(B["origem_sql"]).read_text(encoding="utf-8")
    with conn.cursor() as cur:
        cur.execute(sql)              
    conn.commit()
    with conn.cursor() as cur:
        cur.execute("SET search_path TO public")
    conn.commit()
    print("Origem carregada.")
else:
    print("Origem já carregada (use recarregar_origem: true no config.yaml para refazer).")

with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM campo_real_energia.medicao_geracao")
    print("medicao_geracao:", cur.fetchone()[0], "linhas (esperado: 78798)")

Origem já carregada (use recarregar_origem: true no config.yaml para refazer).
medicao_geracao: 78798 linhas (esperado: 78798)


## 3. Schema `externo`: log, raw e tratada

- **`log_extracao`**: uma linha por execução (início, fim, linhas lidas, linhas gravadas, status, erro).
- **`raw_pld_horario`**: cada registro da API em `jsonb`, exatamente como chegou, com data/hora da extração, origem (URL) e offset da página.
- **`pld_horario`** (tratada): grão **1 submercado × 1 hora**, PK `(data_hora, cod_submercado)`.
- **`pld_horario_rejeitado`**: registros da raw que não passaram no tratamento, com o motivo (insumo para a Etapa 2).

A tratada não tem FK para `campo_real_energia.submercado` de propósito: o script de origem faz `DROP SCHEMA ... CASCADE` e derrubaria a FK. O domínio é garantido por `CHECK`.

In [6]:
DDL = """
CREATE SCHEMA IF NOT EXISTS externo;

CREATE TABLE IF NOT EXISTS externo.log_extracao (
  id_log          BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  etapa           VARCHAR(12) NOT NULL DEFAULT 'EXTRACAO',
  fonte           VARCHAR(30) NOT NULL,
  recurso         VARCHAR(80) NOT NULL,
  inicio          TIMESTAMPTZ NOT NULL DEFAULT now(),
  fim             TIMESTAMPTZ,
  linhas_lidas    INTEGER DEFAULT 0,
  linhas_gravadas INTEGER DEFAULT 0,
  status          VARCHAR(15) NOT NULL DEFAULT 'EXECUTANDO',
  erro            TEXT,
  observacao      TEXT
);
ALTER TABLE externo.log_extracao ADD COLUMN IF NOT EXISTS etapa VARCHAR(12) NOT NULL DEFAULT 'EXTRACAO';
ALTER TABLE externo.log_extracao ADD COLUMN IF NOT EXISTS observacao TEXT;

CREATE TABLE IF NOT EXISTS externo.raw_pld_horario (
  id_raw        BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  resource_id   VARCHAR(40) NOT NULL,
  ano           SMALLINT    NOT NULL,
  extraido_em   TIMESTAMPTZ NOT NULL,
  origem_url    TEXT        NOT NULL,
  offset_pagina INTEGER     NOT NULL,
  payload       JSONB       NOT NULL
);
CREATE INDEX IF NOT EXISTS idx_raw_pld_resource ON externo.raw_pld_horario (resource_id);

CREATE TABLE IF NOT EXISTS externo.pld_horario (
  data_hora      TIMESTAMP     NOT NULL,                 -- horário de brasília
  cod_submercado CHAR(2)       NOT NULL CHECK (cod_submercado IN ('SE','S','NE','N')),
  pld_rs_mwh     NUMERIC(10,2) NOT NULL,
  resource_id    VARCHAR(40)   NOT NULL,
  extraido_em    TIMESTAMPTZ   NOT NULL,
  carregado_em   TIMESTAMPTZ   NOT NULL DEFAULT now(),
  PRIMARY KEY (data_hora, cod_submercado)
);

CREATE TABLE IF NOT EXISTS externo.pld_horario_rejeitado (
  id_rejeitado  BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_raw        BIGINT,
  resource_id   VARCHAR(40) NOT NULL,
  motivo        VARCHAR(40) NOT NULL,
  payload       JSONB,
  registrado_em TIMESTAMPTZ NOT NULL DEFAULT now()
);
"""
with conn.cursor() as cur:
    cur.execute(DDL)
conn.commit()

def abrir_log(conn, recurso, etapa="EXTRACAO", fonte="CCEE_PLD"):
    with conn.cursor() as cur:
        cur.execute("INSERT INTO externo.log_extracao (etapa, fonte, recurso) VALUES (%s,%s,%s) RETURNING id_log",
                    (etapa, fonte, str(recurso)[:80]))
        id_log = cur.fetchone()[0]
    conn.commit()
    return id_log

def fechar_log(conn, id_log, lidas, gravadas, status, erro=None, obs=None):
    with conn.cursor() as cur:
        cur.execute("""UPDATE externo.log_extracao
                       SET fim=now(), linhas_lidas=%s, linhas_gravadas=%s, status=%s, erro=%s, observacao=%s
                       WHERE id_log=%s""", (lidas, gravadas, status, erro, obs, id_log))
    conn.commit()

print("Schema externo pronto.")

Schema externo pronto.


## 4. Cliente HTTP para a API da CCEE

A API da CCEE respondeu **403 "Acesso bloqueado"** a clientes simples (`requests`) tanto no Colab quanto localmente. O cliente abaixo tenta, em ordem, os *backends* do `config.yaml`:

1. `curl_cffi`: imita o handshake TLS de um Chrome real;
2. `requests`: com headers de navegador;
3. `playwright` (opcional, plano C): navegador de verdade.

Em caso de 403, passa direto para o próximo backend (repetir não adianta). Timeout, 429 e 5xx têm retry com backoff exponencial; o backend que funcionou é reaproveitado nas chamadas seguintes.

In [7]:
CKAN = C["base_url"]
REFERER = "https://dadosabertos.ccee.org.br/dataset/pld_horario"

def _headers():
    return {"User-Agent": H["user_agent"], "Accept": "application/json, text/plain, */*",
            "Accept-Language": H["accept_language"], "Referer": REFERER}

def _get_requests(url, params):
    r = requests.get(url, params=params, headers=_headers(), timeout=H["timeout"])
    return r.status_code, r.headers, r.text, r.url

def _get_curl_cffi(url, params):
    from curl_cffi import requests as cr
    r = cr.get(url, params=params, headers=_headers(), impersonate=H["impersonate"], timeout=H["timeout"])
    return r.status_code, r.headers, r.text, str(r.url)

def _get_playwright(url, params):
    # plano C. Roda em outra thread porque a API síncrona do Playwright não funciona dentro do loop do Jupyter.
    from concurrent.futures import ThreadPoolExecutor
    from urllib.parse import urlencode
    full = f"{url}?{urlencode(params)}" if params else url
    def job():
        from playwright.sync_api import sync_playwright
        with sync_playwright() as p:
            b = p.chromium.launch(headless=True)
            ctx = b.new_context(locale="pt-BR")
            page = ctx.new_page()
            page.goto(REFERER, timeout=60000)               # cookies
            resp = ctx.request.get(full, timeout=60000)
            out = (resp.status, dict(resp.headers), resp.text(), full)
            b.close()
            return out
    with ThreadPoolExecutor(1) as ex:
        return ex.submit(job).result()

BACKENDS = {"curl_cffi": _get_curl_cffi, "requests": _get_requests, "playwright": _get_playwright}
_backend_ativo = None

def _espera(headers, padrao):
    try:
        return int(headers.get("Retry-After", padrao))
    except (TypeError, ValueError):
        return padrao

def ckan_get(action, params=None):
    """Chama uma ação CKAN. Retorna (result, url_final). Levanta RuntimeError se tudo falhar."""
    global _backend_ativo
    url = f"{CKAN}/{action}"
    ordem = [_backend_ativo] if _backend_ativo else list(H["backends"])
    ultimo_erro = None
    for backend in ordem:
        for t in range(1, H["max_tentativas"] + 1):
            padrao = H["backoff_segundos"] * 2 ** (t - 1)
            try:
                status, headers, texto, url_final = BACKENDS[backend](url, params or {})
            except ImportError as e:
                ultimo_erro = f"{backend}: não instalado ({e})"
                break
            except Exception as e:                           # timeout, conexão, TLS
                ultimo_erro = f"{backend}: {type(e).__name__}: {e}"
                time.sleep(padrao)
                continue
            if status == 403:
                ultimo_erro = f"{backend}: 403 bloqueado ({texto[:120]!r})"
                break                                        # próximo backend
            if status == 429 or status >= 500:
                ultimo_erro = f"{backend}: HTTP {status}"
                time.sleep(_espera(headers, padrao))
                continue
            if status != 200:
                raise RuntimeError(f"{backend}: HTTP {status} em {url_final}")
            try:
                j = json.loads(texto)
            except ValueError:
                ultimo_erro = f"{backend}: resposta não-JSON ({texto[:120]!r})"
                time.sleep(padrao)
                continue
            if not j.get("success"):
                raise ValueError(f"CKAN success=false: {j.get('error')}")
            _backend_ativo = backend
            return j["result"], url_final
    _backend_ativo = None
    raise RuntimeError(f"Falha após tentativas. Último erro: {ultimo_erro}")

def diagnostico():
    for b in H["backends"]:
        try:
            s, h, t, u = BACKENDS[b](f"{CKAN}/status_show", {})
            print(f"{b:10} -> HTTP {s} | {t[:90]!r}")
        except Exception as e:
            print(f"{b:10} -> {type(e).__name__}: {e}")

diagnostico()

curl_cffi  -> HTTP 200 | '{"help": "https://dadosabertos.ccee.org.br/api/3/action/help_show?name=status_show", "succ'
requests   -> HTTP 403 | '<meta charset="UTF-8"><title>Acesso bloqueado</title><link rel=icon href=data:,>\n<style>bo'


**Leia a saída acima.** Se algum backend retornou `HTTP 200` com JSON, siga em frente.
Se todos retornaram 403, teste o link no navegador, tente outra rede (4G, sem VPN) e, como último recurso, acrescente `playwright` em `backends` no `config.yaml` (`%pip install playwright` e `playwright install chromium`). Registre o que aconteceu no README: o comportamento do processo com a fonte bloqueada também é avaliado.

## 5. Descobrir os recursos e extrair (camada raw)

In [8]:
ANO_RE = re.compile(r"(?<!\d)(20\d{2})(?!\d)")

def descobrir_recursos():
    """{ano: resource_id}. Config manda; o package_show completa o que faltar."""
    ids = {int(k): v for k, v in (C.get("resource_ids") or {}).items()}
    try:
        res, _ = ckan_get("package_show", {"id": C["dataset_id"]})
    except Exception as e:
        print("package_show falhou:", e)
        print("Preencha ccee.resource_ids no config.yaml com os UUIDs da página do dataset.")
        return ids
    print(f"{'resource_id':38} {'datastore':9} nome")
    for r in res["resources"]:
        print(f"{r['id']:38} {str(r.get('datastore_active')):9} {r.get('name')}")
        anos = ANO_RE.findall(f"{r.get('name') or ''} {r.get('description') or ''}")
        if len(set(anos)) == 1 and int(anos[0]) not in ids:
            ids[int(anos[0])] = r["id"]
    return ids

def extrair_recurso(conn, resource_id, ano):
    id_log = abrir_log(conn, resource_id, "EXTRACAO")
    lidas = gravadas = 0
    try:
        extraido_em = datetime.now(timezone.utc)
        offset, total, campos_ref, avisos = 0, None, None, []
        with conn.cursor() as cur:
            # recarga idempotente: apaga e regrava na MESMA transação
            # (se algo falhar, rollback e a raw anterior fica preservada)
            cur.execute("DELETE FROM externo.raw_pld_horario WHERE resource_id=%s", (resource_id,))
            while True:
                res, url = ckan_get("datastore_search",
                                    {"resource_id": resource_id, "limit": C["page_size"], "offset": offset})
                recs, total = res["records"], res.get("total")
                if not recs:
                    break                                    # fim da paginação
                campos = [f["id"] for f in res["fields"]]
                if campos_ref is None:
                    campos_ref = campos
                elif campos != campos_ref:
                    avisos.append(f"colunas mudaram no offset {offset}")
                    print(f"AVISO: colunas mudaram no offset {offset}: {campos}")
                lidas += len(recs)
                execute_values(cur,
                    """INSERT INTO externo.raw_pld_horario
                       (resource_id, ano, extraido_em, origem_url, offset_pagina, payload) VALUES %s""",
                    [(resource_id, ano, extraido_em, url, offset, Json(r)) for r in recs], page_size=1000)
                gravadas += len(recs)
                offset += len(recs)
                print(f"  {ano}: {gravadas}/{total}")
                if total is not None and offset >= total:
                    break
                time.sleep(H["pausa_entre_paginas"])
        if lidas == 0:
            raise ValueError("resposta vazia: recurso sem registros")
        if total is not None and gravadas != total:
            raise ValueError(f"gravadas={gravadas} difere do total informado pela API ({total})")
        conn.commit()
        fechar_log(conn, id_log, lidas, gravadas, "SUCESSO", obs="; ".join(avisos) or None)
    except Exception as e:
        conn.rollback()
        fechar_log(conn, id_log, lidas, 0, "ERRO", erro=str(e)[:500])
        raise

recursos = descobrir_recursos()
print("\nrecursos usados:", {a: recursos.get(a) for a in C["anos"]})

resource_id                            datastore nome
7c90a379-5e98-46ff-a11b-9120bcf81ac4   True      pld_historico_semanal_2001_2020
51922462-16b4-4c64-8327-4e14d6ee8c6c   True      pld_horario_2021
723cf7e6-6c29-4da6-aa39-e4c8804baf65   True      pld_horario_2022
5fc317af-7191-4f8a-94e7-f77c56c747b3   True      pld_horario_2023
1b5b6946-8036-4622-a7a3-b21f33fc52b7   True      pld_horario_2024
2a180a6b-f092-43eb-9f82-a48798b803dc   True      pld_horario_2025
3f279d6b-1069-42f7-9b0a-217b084729c4   True      pld_horario_2026

recursos usados: {2025: '2a180a6b-f092-43eb-9f82-a48798b803dc', 2026: '3f279d6b-1069-42f7-9b0a-217b084729c4'}


In [9]:
for ano in C["anos"]:
    rid = recursos.get(ano)
    if not rid:
        id_log = abrir_log(conn, f"pld_horario_{ano}", "EXTRACAO")
        fechar_log(conn, id_log, 0, 0, "ERRO", erro=f"recurso de {ano} não encontrado no dataset")
        print(f"{ano}: recurso não encontrado")
        continue
    try:
        print(f"Extraindo {ano} ({rid})")
        extrair_recurso(conn, rid, ano)
        print(f"{ano}: OK")
    except Exception as e:
        print(f"{ano}: ERRO -> {e}")

Extraindo 2025 (2a180a6b-f092-43eb-9f82-a48798b803dc)
  2025: 5000/35040
  2025: 10000/35040
  2025: 15000/35040
  2025: 20000/35040
  2025: 25000/35040
  2025: 30000/35040
  2025: 35000/35040
  2025: 35040/35040
2025: OK
Extraindo 2026 (3f279d6b-1069-42f7-9b0a-217b084729c4)
  2026: 5000/29952
  2026: 10000/29952
  2026: 15000/29952
  2026: 20000/29952
  2026: 25000/29952
  2026: 29952/29952
2026: OK


### Conferir como o dado chegou

Veja os nomes de coluna e alguns registros brutos. Se os nomes forem diferentes dos candidatos do `config.yaml` (`tratamento.campos`), ajuste lá antes de tratar.

In [10]:
with conn.cursor() as cur:
    cur.execute("""SELECT resource_id, ano, COUNT(*), MIN(extraido_em)
                   FROM externo.raw_pld_horario GROUP BY 1,2 ORDER BY 2""")
    for row in cur.fetchall():
        print(row)
    cur.execute("SELECT payload FROM externo.raw_pld_horario ORDER BY id_raw LIMIT 5")
    amostra = [r[0] for r in cur.fetchall()]
display(pd.DataFrame(amostra))

('2a180a6b-f092-43eb-9f82-a48798b803dc', 2025, 35040, datetime.datetime(2026, 11, 8, 19, 44, 50, 503648, tzinfo=datetime.timezone.utc))
('3f279d6b-1069-42f7-9b0a-217b084729c4', 2026, 29952, datetime.datetime(2026, 11, 8, 19, 44, 59, 693080, tzinfo=datetime.timezone.utc))


,DIA,_id,HORA,PLD_HORA,SUBMERCADO,MES_REFERENCIA,PERIODO_COMERCIALIZACAO
0,31,1,0,205.25,NORDESTE,202512,721
1,31,2,0,205.26,NORTE,202512,721
2,31,3,0,205.26,SUDESTE,202512,721
3,31,4,0,205.25,SUL,202512,721
4,31,5,1,197.52,NORDESTE,202512,722


## 6. Tratamento (raw → `externo.pld_horario`)

Regras desta camada (apenas **forma**, sem inventar dado):

- converte `data/hora` (coluna única, ou `DIA` + `HORA` [+ `MES_REFERENCIA`]), ajustando hora 1-24 para 0-23 quando for o caso;
- padroniza o submercado para `SE`, `S`, `NE`, `N`;
- converte o PLD para número (aceita `1.234,56` e `1234.56`);
- registros inválidos ou duplicados vão para `pld_horario_rejeitado` com o motivo. Nada é descartado em silêncio.

In [11]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().upper()

SUBMERCADOS = {
    "SUDESTE": "SE", "SUDESTE/CENTRO-OESTE": "SE", "SUDESTE E CENTRO-OESTE": "SE", "SE/CO": "SE", "SE": "SE",
    "SUL": "S", "S": "S", "NORDESTE": "NE", "NE": "NE", "NORTE": "N", "N": "N",
}

def achar_coluna(cols, candidatos):
    mapa = {norm(c): c for c in cols}
    for cand in candidatos:
        if norm(cand) in mapa:
            return mapa[norm(cand)]
    return None

def para_numero(serie):
    s = serie.astype(str).str.replace(r"\s", "", regex=True)
    ptbr = s.str.contains(",", regex=False)
    s = s.where(~ptbr, s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(s, errors="coerce")

def montar_data_hora(df, cm):
    if cm["data_hora"]:
        return pd.to_datetime(df[cm["data_hora"]], errors="coerce")
    if not (cm["dia"] and cm["hora"]):
        raise ValueError(f"não consegui montar data_hora. Colunas: {list(df.columns)}")
    hora = pd.to_numeric(df[cm["hora"]].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    base = str(T.get("hora_base", "auto"))
    if base == "1" or (base == "auto" and hora.min() >= 1):
        hora = hora - 1                                       # 1-24 -> 0-23
        print("  hora_base: 1-24 detectada; subtraindo 1")
    dia_txt = df[cm["dia"]].astype(str).str.strip()
    if dia_txt.str.contains(r"[-/]", regex=True).any():       # DIA é uma data completa
        data = pd.to_datetime(dia_txt, errors="coerce", dayfirst=dia_txt.str.contains("/").any())
    elif cm["mes_referencia"]:                                # DIA é o dia do mês
        mes = df[cm["mes_referencia"]].astype(str).str.extract(r"(\d{6})")[0]
        data = pd.to_datetime(mes + dia_txt.str.extract(r"(\d+)")[0].str.zfill(2),
                              format="%Y%m%d", errors="coerce")
    else:
        raise ValueError("DIA é só o número do dia e não achei MES_REFERENCIA. Ajuste tratamento.campos.")
    return data + pd.to_timedelta(hora, unit="h")

def tratar_recurso(conn, resource_id):
    id_log = abrir_log(conn, resource_id, "TRATAMENTO")
    lidas = gravadas = 0
    try:
        with conn.cursor() as cur:
            cur.execute("""SELECT id_raw, extraido_em, payload FROM externo.raw_pld_horario
                           WHERE resource_id=%s ORDER BY id_raw""", (resource_id,))
            linhas = cur.fetchall()
        if not linhas:
            raise ValueError("raw vazia para este recurso")
        extraido_em = linhas[0][1]
        df = pd.DataFrame([l[2] for l in linhas])
        lidas = len(df)
        cols = list(df.columns)
        cm = {k: achar_coluna(cols, v) for k, v in T["campos"].items()}
        print("  mapeamento de colunas:", cm)
        if cm["submercado"] is None or cm["pld"] is None:
            raise ValueError(f"não mapeei submercado/PLD. Colunas da API: {cols}")

        df["_ts"] = montar_data_hora(df, cm)
        df["_sub"] = df[cm["submercado"]].map(lambda v: SUBMERCADOS.get(norm(v)))
        df["_pld"] = para_numero(df[cm["pld"]])

        motivo = pd.Series(None, index=df.index, dtype=object)
        motivo[df["_ts"].isna()] = "data_hora_invalida"
        motivo[motivo.isna() & df["_sub"].isna()] = "submercado_desconhecido"
        motivo[motivo.isna() & df["_pld"].isna()] = "pld_invalido"
        ok = df[motivo.isna()]
        dup = ok.duplicated(["_ts", "_sub"], keep="last")      # mantém o registro mais recente
        motivo[ok.index[dup]] = "duplicado"

        boas, rej = df[motivo.isna()], df[motivo.notna()]
        rows = [(t.to_pydatetime(), s, round(float(p), 2), resource_id, extraido_em)
                for t, s, p in zip(boas["_ts"], boas["_sub"], boas["_pld"])]
        rows_rej = [(linhas[i][0], resource_id, motivo[i], Json(linhas[i][2])) for i in rej.index]

        with conn.cursor() as cur:
            cur.execute("DELETE FROM externo.pld_horario WHERE resource_id=%s", (resource_id,))
            cur.execute("DELETE FROM externo.pld_horario_rejeitado WHERE resource_id=%s", (resource_id,))
            if rows:
                execute_values(cur, """
                    INSERT INTO externo.pld_horario (data_hora, cod_submercado, pld_rs_mwh, resource_id, extraido_em)
                    VALUES %s
                    ON CONFLICT (data_hora, cod_submercado) DO UPDATE
                      SET pld_rs_mwh=EXCLUDED.pld_rs_mwh, resource_id=EXCLUDED.resource_id,
                          extraido_em=EXCLUDED.extraido_em, carregado_em=now()""", rows, page_size=1000)
            if rows_rej:
                execute_values(cur, """INSERT INTO externo.pld_horario_rejeitado
                                       (id_raw, resource_id, motivo, payload) VALUES %s""", rows_rej, page_size=1000)
        conn.commit()
        gravadas = len(rows)
        resumo = motivo.value_counts().to_dict()
        fechar_log(conn, id_log, lidas, gravadas, "SUCESSO",
                   obs=f"rejeitadas={len(rej)} {resumo}" if len(rej) else None)
        print(f"  lidas={lidas} gravadas={gravadas} rejeitadas={len(rej)} {resumo if len(rej) else ''}")
    except Exception as e:
        conn.rollback()
        fechar_log(conn, id_log, lidas, 0, "ERRO", erro=str(e)[:500])
        raise

with conn.cursor() as cur:
    cur.execute("SELECT DISTINCT resource_id, ano FROM externo.raw_pld_horario ORDER BY ano")
    para_tratar = cur.fetchall()

for rid, ano in para_tratar:
    try:
        print(f"Tratando {ano} ({rid})")
        tratar_recurso(conn, rid)
    except Exception as e:
        print(f"{ano}: ERRO -> {e}")

Tratando 2025 (2a180a6b-f092-43eb-9f82-a48798b803dc)
  mapeamento de colunas: {'submercado': 'SUBMERCADO', 'pld': 'PLD_HORA', 'data_hora': None, 'dia': 'DIA', 'hora': 'HORA', 'mes_referencia': 'MES_REFERENCIA'}
  lidas=35040 gravadas=35040 rejeitadas=0 
Tratando 2026 (3f279d6b-1069-42f7-9b0a-217b084729c4)
  mapeamento de colunas: {'submercado': 'SUBMERCADO', 'pld': 'PLD_HORA', 'data_hora': None, 'dia': 'DIA', 'hora': 'HORA', 'mes_referencia': 'MES_REFERENCIA'}
  lidas=29952 gravadas=29952 rejeitadas=0 


## 7. Verificação

In [12]:
def consulta(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

# completude: quantas horas deveriam existir entre a primeira e a última hora de cada submercado/ano?
display(consulta("""
    SELECT EXTRACT(YEAR FROM data_hora)::INT AS ano, cod_submercado,
           COUNT(*) AS horas, MIN(data_hora) AS primeira, MAX(data_hora) AS ultima,
           (EXTRACT(EPOCH FROM (MAX(data_hora) - MIN(data_hora)))/3600)::INT + 1 AS horas_esperadas,
           ROUND(AVG(pld_rs_mwh), 2) AS pld_medio, MIN(pld_rs_mwh) AS pld_min, MAX(pld_rs_mwh) AS pld_max
    FROM externo.pld_horario
    GROUP BY 1, 2 ORDER BY 1, 2
"""))
print("Referência: 2025 completo = 8760 horas por submercado (35040 no total).")

print("\nRejeitados por motivo:")
display(consulta("SELECT resource_id, motivo, COUNT(*) FROM externo.pld_horario_rejeitado GROUP BY 1,2 ORDER BY 1,2"))

,ano,cod_submercado,horas,primeira,ultima,horas_esperadas,pld_medio,pld_min,pld_max
0,2025,N,8760,2025-01-01,2025-12-31 23:00:00,8760,182.83,58.60,1421.87
1,2025,NE,8760,2025-01-01,2025-12-31 23:00:00,8760,177.16,58.60,1421.86
2,2025,S,8760,2025-01-01,2025-12-31 23:00:00,8760,226.83,58.60,1421.87
3,2025,SE,8760,2025-01-01,2025-12-31 23:00:00,8760,224.31,58.60,1421.87
4,2026,N,7488,2026-01-01,2026-11-08 23:00:00,7488,179.29,57.31,1611.04
5,2026,NE,7488,2026-01-01,2026-11-08 23:00:00,7488,178.43,57.31,1611.04
6,2026,S,7488,2026-01-01,2026-11-08 23:00:00,7488,218.78,57.31,1611.04
7,2026,SE,7488,2026-01-01,2026-11-08 23:00:00,7488,197.55,57.31,1611.04


Referência: 2025 completo = 8760 horas por submercado (35040 no total).

Rejeitados por motivo:


,resource_id,motivo,count


In [13]:
display(consulta("""SELECT id_log, etapa, recurso, inicio, fim, linhas_lidas, linhas_gravadas, status, erro, observacao
                    FROM externo.log_extracao ORDER BY id_log DESC LIMIT 20"""))

,id_log,etapa,recurso,inicio,fim,linhas_lidas,linhas_gravadas,status,erro,observacao
0,4,TRATAMENTO,3f279d6b-1069-42f7-9b0a-217b084729c4,2026-11-08 19:46:32.359422+00:00,2026-11-08 19:46:36.434460+00:00,29952,29952,SUCESSO,None,None
1,3,TRATAMENTO,2a180a6b-f092-43eb-9f82-a48798b803dc,2026-11-08 19:45:15.190808+00:00,2026-11-08 19:46:32.295401+00:00,35040,35040,SUCESSO,None,None
2,2,EXTRACAO,3f279d6b-1069-42f7-9b0a-217b084729c4,2026-11-08 19:45:00.361345+00:00,2026-11-08 19:45:07.424020+00:00,29952,29952,SUCESSO,None,None
3,1,EXTRACAO,2a180a6b-f092-43eb-9f82-a48798b803dc,2026-11-08 19:44:51.169581+00:00,2026-11-08 19:45:00.315078+00:00,35040,35040,SUCESSO,None,None
